# MINI Cells — Experiment 019: Proposal Utility Discovery

Discover whether label-free local or boundary observables predict the true marginal utility of a sleeping capability-tissue proposal across held-out skill families. The experiment first validates the continuous recruitment oracle, then trains six one-cell donor tissues per replicate and performs strong leave-one-family-out evaluation.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
ROOT = Path('/kaggle/working/mini-cells')
os.chdir('/kaggle/working')
if not (ROOT / '.git').exists():
    if ROOT.exists(): shutil.rmtree(ROOT)
    subprocess.run(['git','clone','--depth','1','https://github.com/ArcheLabs/mini-cells.git',str(ROOT)], check=True)
else:
    subprocess.run(['git','fetch','origin'], cwd=ROOT, check=True)
    subprocess.run(['git','switch','main'], cwd=ROOT, check=True)
    subprocess.run(['git','reset','--hard','origin/main'], cwd=ROOT, check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[lm]'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('repo:', ROOT)
subprocess.run(['git','rev-parse','HEAD'], check=True)


In [ ]:
import torch
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu_count': torch.cuda.device_count(), 'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
if torch.cuda.device_count() < 1:
    raise RuntimeError('Experiment 019 requires a Kaggle GPU accelerator')


## Preflight invariants

The 019 tests enforce `e=0 -> exact Phase-1`, `e=1 -> exact fully-active donor`, gradient/finite-difference oracle agreement on a small fixture, target-free feature invariance, and the multi-family corpus. Regression tests cover the 018b/017/016 mechanisms reused by 019.


In [ ]:
tests = [
    'tests/test_language_proposal_utility.py',
    'tests/test_language_pressure_recruitment.py',
    'tests/test_language_localized_learning.py',
    'tests/test_language_growing_organism.py',
]
subprocess.run([sys.executable,'-m','pytest',*tests,'-q'], cwd=ROOT, check=True)


## Full 3-replicate run

Two physical GPUs are used concurrently when available. Each replicate trains one Phase-1 organism, six one-cell skill donors plus one untrained control, then measures example-level proposal utilities. The parent process performs strong leave-one-family-out estimator evaluation.


In [ ]:
OUT = ROOT / 'results' / 'proposal-utility-discovery-v1'
subprocess.run([sys.executable,'scripts/run_proposal_utility_discovery.py'], cwd=ROOT, check=True)
print('results:', OUT)


In [ ]:
import json, pandas as pd
from IPython.display import Image, Markdown, display
decision = json.loads((OUT / 'decision.json').read_text(encoding='utf-8'))
display(Markdown(f"## {decision['status']}"))
display(Markdown(f"**Question:** {decision['question']}"))
display(pd.DataFrame([decision['results']]))
display(pd.read_csv(OUT / 'oracle-consistency.csv'))
display(pd.read_csv(OUT / 'donor-summary.csv').head(30))
display(pd.read_csv(OUT / 'estimator-results.csv'))
for name in ['oracle-gradient-vs-fd.png','candidate-utility-matrix.png','heldout-spearman.png','heldout-auc.png','heldout-top1.png','heldout-regret.png','feature-oracle-correlations.png']:
    path = OUT / name
    if path.exists(): display(Image(filename=str(path)))


## Curate and publish

Publication defaults to enabled to match the current research workflow. Set `MINICELLS_PUBLISH=0` before this cell if you only want local Kaggle artifacts. The publisher writes the locked result set to `kaggle/experiment-019-results`.


In [ ]:
publish = os.environ.get('MINICELLS_PUBLISH', '1').strip().lower() not in {'0','false','no'}
cmd = [sys.executable, 'scripts/publish_experiment_019_results.py']
if publish: cmd.append('--push')
subprocess.run(cmd, cwd=ROOT, check=True)
